# 10) M1.5 Visualising energy data — Exercises

**Goals:** consistent carrier colours, DST-safe daily profiles and year heatmaps, duration curves and residual load, a reusable dispatch plot that is checked by its data, the merit-order steps behind the prices, and saving figures for your portfolio.

The setup cell rebuilds the lecture's **synthetic** year and dispatch (takes a few seconds). Plot functions return their data along with the figure, so the tests can check what the eye cannot. Run the notebook in Jupyter to see the figures (remove the `matplotlib.use("Agg")` line if no figures appear).

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import math
import tempfile
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")                      # figures are checked through their data in this exercise
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pvlib
import pypsa
from scipy import stats

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)


CARRIER_COLORS = {"nuclear": "#7B4F9E", "hydro": "#2A7AB8", "wind": "#4FB0C6", "solar": "#F2C14E",
                  "coal": "#5C4033", "gas": "#E07B39", "imports": "#9E9E9E", "load": "#000000"}


def power_curve(v, v_cut_in=3.0, v_rated=12.0, v_cut_out=25.0):
    """Generic per-unit turbine output (M1.4)."""
    v = np.asarray(v, dtype=float)
    rising = (v ** 3 - v_cut_in ** 3) / (v_rated ** 3 - v_cut_in ** 3)
    out = np.where((v >= v_cut_in) & (v < v_rated), rising, 0.0)
    return np.where((v >= v_rated) & (v <= v_cut_out), 1.0, out)


def synthetic_year(year: int = 2025, seed: int = 3) -> pd.DataFrame:
    """SYNTHETIC hourly year (UTC, interval start): load in MW, wind and solar availability in p.u.

    Built from the physics of M1.4 (power curve, clear-sky sun) plus random weather.
    Plausible in shape for a Croatia-like system, but NOT measured data.
    """
    rng = np.random.default_rng(seed)
    idx = pd.date_range(f"{year}-01-01", f"{year + 1}-01-01", freq="h", tz="UTC", inclusive="left")
    local = idx.tz_convert("Europe/Zagreb")
    doy, hour = local.dayofyear.to_numpy(), local.hour.to_numpy()
    weekend = local.dayofweek.to_numpy() >= 5
    load = (2100 + 250 * np.cos(2 * np.pi * (doy - 15) / 365)          # winter peak
            + 350 * np.sin((hour - 7) / 24 * 2 * np.pi) - 200 * weekend
            + rng.normal(0, 40, len(idx)))
    z = np.empty(len(idx)); z[0] = 0.0                                  # persistent weather
    for t in range(1, len(idx)):
        z[t] = 0.97 * z[t - 1] + np.sqrt(1 - 0.97 ** 2) * rng.normal()
    season = 1 + 0.15 * np.cos(2 * np.pi * (doy - 15) / 365)            # windier winters
    speeds = stats.weibull_min.ppf(stats.norm.cdf(z), 2.0, scale=7.5) * season
    clouds = np.empty(len(idx)); clouds[0] = 0.8
    for t in range(1, len(idx)):
        clouds[t] = np.clip(0.95 * clouds[t - 1] + 0.05 * 0.75 + rng.normal(0, 0.05), 0.15, 1.0)
    zen = pvlib.solarposition.get_solarposition(idx + pd.Timedelta("30min"), 43.51, 16.44)["apparent_zenith"]
    ghi_clear = pvlib.clearsky.haurwitz(zen)["ghi"].fillna(0).to_numpy()
    return pd.DataFrame({"load_MW": load.round(1), "wind_pu": power_curve(speeds).round(3),
                         "solar_pu": np.clip(ghi_clear / 1000 * 0.86 * clouds, 0, 1).round(3)}, index=idx)


def build_and_solve(profiles: pd.DataFrame) -> pypsa.Network:
    """M1.2's stylised single-bus system, dispatched for the given profiles (naive UTC snapshots)."""
    n = pypsa.Network()
    n.set_snapshots(profiles.index.tz_localize(None))
    n.add("Carrier", "AC")
    n.add("Bus", "HR", carrier="AC")
    for name, co2 in [("nuclear", 0.0), ("hydro", 0.0), ("wind", 0.0), ("solar", 0.0),
                      ("coal", 0.34), ("gas", 0.20), ("imports", 0.0)]:
        n.add("Carrier", name, co2_emissions=co2)
    n.add("Load", "demand", bus="HR", carrier="AC", p_set=profiles["load_MW"].to_numpy())
    for name, carrier, p_nom, mc, eff in [("nuclear_share", "nuclear", 350, 10, 0.33),
                                          ("hydro_ror", "hydro", 1500, 2, 1.0),
                                          ("wind", "wind", 1000, 0, 1.0), ("solar", "solar", 800, 0, 1.0),
                                          ("coal", "coal", 210, 95, 0.38), ("gas_ccgt", "gas", 700, 110, 0.58),
                                          ("gas_ocgt", "gas", 300, 160, 0.38),
                                          ("imports", "imports", 1500, 120, 1.0)]:
        n.add("Generator", name, bus="HR", carrier=carrier, p_nom=p_nom, marginal_cost=mc, efficiency=eff)
    n.generators_t.p_max_pu["hydro_ror"] = 0.45
    n.generators_t.p_max_pu["wind"] = profiles["wind_pu"].to_numpy()
    n.generators_t.p_max_pu["solar"] = profiles["solar_pu"].to_numpy()
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n



profiles = synthetic_year()          # synthetic, as in the lecture
n = build_and_solve(profiles)

### Warm-ups

**1. Consistent colours**

Return the list of colours for a list of carriers, using `CARRIER_COLORS` and a neutral grey `"#BBBBBB"` for anything unknown — never let the plotting library pick colours for carriers.

In [ ]:
def colors_for(carriers: list[str]) -> list[str]:
    """Colour per carrier from CARRIER_COLORS, grey for unknown carriers."""
    ...  # YOUR CODE HERE

In [ ]:
assert colors_for(["wind", "gas"]) == [CARRIER_COLORS["wind"], CARRIER_COLORS["gas"]]
assert colors_for(["biomass"]) == ["#BBBBBB"], "unknown carriers get grey"

**2. Average daily profile in local time**

Return a DataFrame with the **local** hour (0–23) as index and one column per group, holding the mean of an hourly UTC series. `groups` maps each local month (1–12) to a group name.

In [ ]:
SEASONS = {12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring",
           6: "summer", 7: "summer", 8: "summer", 9: "autumn", 10: "autumn", 11: "autumn"}


def daily_profile(series: pd.Series, groups: dict, tz: str = "Europe/Zagreb") -> pd.DataFrame:
    """Mean value per local hour (rows) and group (columns)."""
    ...  # YOUR CODE HERE

In [ ]:
prof = daily_profile(profiles.solar_pu, SEASONS)
assert prof.shape == (24, 4)
assert set(prof.columns) == {"winter", "spring", "summer", "autumn"}
assert prof.loc[0].max() == 0, "no sun at local midnight"
assert 11 <= prof["summer"].idxmax() <= 13, "summer solar peaks around local noon (UTC would be 10-11)"
assert prof["summer"].max() > prof["winter"].max()

### Core

**3. A DST-safe year matrix**

Return a day × local-hour matrix for one **local** calendar year: missing local hours stay NaN, the doubled autumn hour is averaged, and days outside the local year are dropped (the lecture's heatmap).

In [ ]:
def year_matrix(series: pd.Series, year: int, tz: str = "Europe/Zagreb") -> pd.DataFrame:
    """Rows: local dates of `year`; columns: local hours 0-23."""
    ...  # YOUR CODE HERE

In [ ]:
m = year_matrix(profiles.load_MW, 2025)
assert m.shape == (365, 24)
empty = m.stack(future_stack=True)
empty = set(empty[empty.isna()].index)
assert empty == {(pd.Timestamp("2025-01-01").date(), 0), (pd.Timestamp("2025-03-30").date(), 2)}, \
    "only the spring DST hour and local midnight before the UTC year starts are missing"
fall_back = profiles.load_MW["2025-10-26 00:00":"2025-10-26 01:00"]      # both UTC hours are 02:00 local
assert math.isclose(m.loc[pd.Timestamp("2025-10-26").date(), 2], fall_back.mean()), "doubled hour averaged"

**4. Residual load and duration curves**

Write `residual_load` (load minus *available* wind and solar, MW, from the solved network) and `duration_curve` (values sorted high to low with the share of the year in %). Then check the energy identity from the lecture.

In [ ]:
def residual_load(n: pypsa.Network) -> pd.Series:
    """Load minus available wind and solar (MW) per snapshot."""
    ...  # YOUR CODE HERE


def duration_curve(values) -> pd.DataFrame:
    """Values sorted descending with the share of hours (%) at or above each value."""
    ...  # YOUR CODE HERE

In [ ]:
res = residual_load(n)
load = n.loads_t.p_set["demand"]
assert (res <= load + 1e-9).all(), "renewables can only lower the residual load"
dc_load, dc_res = duration_curve(load), duration_curve(res)
assert dc_load["value"].is_monotonic_decreasing and math.isclose(dc_load["share_of_hours_pct"].iloc[-1], 100)
renewable_MWh = (n.get_switchable_as_dense("Generator", "p_max_pu")[["wind", "solar"]]
                 * n.generators.p_nom[["wind", "solar"]]).sum().sum()
assert math.isclose(dc_load["value"].sum() - dc_res["value"].sum(), renewable_MWh), \
    "area between the curves = available wind + solar energy"
print(f"peak load {dc_load.value.iloc[0]:.0f} MW, peak residual {dc_res.value.iloc[0]:.0f} MW")

**5. A dispatch plot you can trust**

Write `plot_dispatch(n, start, end, order)`: stacked generation **by carrier** in the given order (local time, `CARRIER_COLORS`), load as a line, price as a step line on a second axis, MW and EUR/MWh axis labels. Return `(fig, ax, data)` where `data` is the plotted carrier DataFrame — so the test can check it adds up to the load.

In [ ]:
def plot_dispatch(n: pypsa.Network, start: str, end: str, order: list[str]):
    """Stacked dispatch by carrier with load and price, local time. Returns (fig, ax, data)."""
    ...  # YOUR CODE HERE

In [ ]:
order = ["nuclear", "hydro", "wind", "solar", "coal", "gas", "imports"]
fig, ax, data = plot_dispatch(n, "2025-01-13", "2025-01-19", order)
local = n.snapshots.tz_localize("UTC").tz_convert("Europe/Zagreb")
load_w = pd.Series(n.loads_t.p_set["demand"].to_numpy(), index=local).loc[data.index]
assert list(data.columns) == order, "stack in the given (merit) order"
assert np.allclose(data.sum(axis=1), load_w), "the stack must add up to the load"
assert "MW" in ax.get_ylabel() and "EUR/MWh" in fig.axes[1].get_ylabel()
assert len(data) == 7 * 24, "one local week"
plt.close(fig)

**6. Interpretation**

In the duration curves, the peak residual load is only slightly below the peak load, although wind and solar provide several TWh per year. What does that mean for how much firm capacity (plants or storage that can run on demand) the system needs, and how often that capacity runs?

*Your answer here.*

### Challenge

**7. The steps behind the prices**

The lecture showed that prices fall on steps of residual load. Prove it for every hour: write `merit_order_steps(n)` returning the price thresholds as a DataFrame (`carrier_marginal_cost`, `cumulative_MW`) for all generators **except wind and solar**, sorted by marginal cost, using their (constant) available capacity. Then predict each hour's price from residual load alone and compare with PyPSA.

In [ ]:
def merit_order_steps(n: pypsa.Network) -> pd.DataFrame:
    """Cumulative available capacity (MW) of non-variable generators in merit order."""
    ...  # YOUR CODE HERE

In [ ]:
steps = merit_order_steps(n)
assert steps["carrier_marginal_cost"].is_monotonic_increasing
assert math.isclose(steps["cumulative_MW"].iloc[0], 1500 * 0.45), "hydro first: 675 MW available"

res_values = residual_load(n).to_numpy()
idx = np.searchsorted(steps["cumulative_MW"].to_numpy(), res_values - 1e-6)
predicted = steps["carrier_marginal_cost"].to_numpy()[idx]
actual = n.buses_t.marginal_price["HR"].to_numpy()
print(f"hours where residual load predicts the price exactly: {np.mean(predicted == actual):.1%}")
assert np.mean(predicted == actual) > 0.99

**8. Save figures for your portfolio**

Write `save_figure(fig, stem)`: save the figure as **SVG** (sharp at any size, small, good for READMEs) and as **PNG at 200 dpi** (for slides), with tight bounding boxes. Return both paths.

In [ ]:
def save_figure(fig, stem: str | Path) -> tuple[Path, Path]:
    """Save fig as <stem>.svg and <stem>.png (200 dpi). Returns (svg_path, png_path)."""
    ...  # YOUR CODE HERE

In [ ]:
fig, ax, _ = plot_dispatch(n, "2025-07-07", "2025-07-13", order)
with tempfile.TemporaryDirectory() as tmp:
    svg, png = save_figure(fig, Path(tmp) / "figures" / "dispatch_summer_week")
    assert svg.suffix == ".svg" and png.suffix == ".png"
    assert svg.stat().st_size > 0 and png.stat().st_size > 0
    assert svg.read_text()[:200].lstrip().startswith("<?xml"), "SVG is text you can put in a README"
plt.close(fig)

**9. Misleading energy plots**

Name four plotting choices that can make an energy figure misleading even though the data are correct, and how each would mislead a reader. Use examples from this lesson.

*Your answer here.*